# 3.0 — Reconstruct Dataset: Combine & Resplit 70:15:15

Combines **all images** from train/val/test into one pool, deduplicates by image hash,
then creates a fresh **stratified 70:15:15** split. Core logic lives in `algear.dataset.resplit`.

## Setup

In [ ]:
# @title Install dependencies
!pip install -q ultralytics roboflow loguru typer python-dotenv pyyaml matplotlib

In [ ]:
# @title Mount Drive or clone repo
import os
from pathlib import Path
import sys

!git clone https://github.com/Hndra04/AlGear
PROJECT_DIR = Path("/content/AlGear")

os.chdir(str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR))
print(f"Project root: {PROJECT_DIR}")

In [ ]:
# @title Set Roboflow API key
import os
os.environ["ROBOFLOW_API_KEY"] = ""

from algear.config import ROBOFLOW_DIR, ROBOFLOW_API_KEY
print(f"API key set: {bool(ROBOFLOW_API_KEY)}")
print(f"Dataset at: {ROBOFLOW_DIR}")

In [ ]:
# @title Download dataset (if not already present)
if not ROBOFLOW_DIR.exists():
    from algear.dataset import download_roboflow
    download_roboflow(output_dir=ROBOFLOW_DIR)
else:
    print(f"Dataset already exists at {ROBOFLOW_DIR}")

## 1. Run Resplit

All logic (combine, deduplicate, stratified split, write to disk) is in `algear.dataset.resplit`.

In [ ]:
# @title Run resplit (70:15:15)
from algear.dataset import resplit

OUTPUT_DIR = resplit(
    src_dir=ROBOFLOW_DIR,
    output_dir=PROJECT_DIR / "data" / "processed" / "construction-site-safety-resplit",
    train_ratio=0.70,
    val_ratio=0.15,
    test_ratio=0.15,
    seed=42,
)

## 2. Visualize Class Distribution

In [ ]:
# @title Plot class distribution across new splits
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np
import yaml

with open(OUTPUT_DIR / "data.yaml") as f:
    data_cfg = yaml.safe_load(f)
class_names = data_cfg["names"]
num_classes = len(class_names)

def count_instances(lbl_dir: Path) -> Counter:
    c = Counter()
    for lbl in lbl_dir.glob("*.txt"):
        with open(lbl) as f:
            for line in f:
                c[int(line.strip().split()[0])] += 1
    return c

split_counts = {}
for split_name in ["train", "val", "test"]:
    split_counts[split_name] = count_instances(OUTPUT_DIR / split_name / "labels")

names = [class_names[i] for i in range(num_classes)]
x = np.arange(num_classes)
width = 0.25

fig, ax = plt.subplots(figsize=(12, 5))
for i, (split_name, color) in enumerate([("train", "#c0392b"), ("val", "#2980b9"), ("test", "#27ae60")]):
    counts = [split_counts[split_name].get(c, 0) for c in range(num_classes)]
    bars = ax.bar(x + i * width, counts, width, label=split_name, color=color, alpha=0.8)
    for bar, c in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                str(c), ha="center", va="bottom", fontsize=8)

ax.set_xticks(x + width)
ax.set_xticklabels(names, rotation=30, ha="right")
ax.set_ylabel("Instance count")
ax.set_title("Class Distribution Across New Splits (70:15:15)")
ax.legend()
plt.tight_layout()
plt.show()

print(f"{'Class':<15s} {'Train':>8s} {'Val':>8s} {'Test':>8s}")
print("-" * 42)
for cls_id in range(num_classes):
    t = split_counts["train"].get(cls_id, 0)
    v = split_counts["val"].get(cls_id, 0)
    te = split_counts["test"].get(cls_id, 0)
    print(f"{class_names[cls_id]:<15s} {t:>8d} {v:>8d} {te:>8d}")

## 3. Usage

Use the resplit dataset in training:

```python
from algear.modeling.train import train_yolov8

results = train_yolov8(
    data_yaml=OUTPUT_DIR / "data.yaml",
    model_name="yolov8s.pt",
    name="baseline-resplit",
    epochs=50,
    device="0",
)
```